# Withings API — comprehensive fitness data exploration

This notebook exports every applicable available family of Withings data (activity, workouts, body measurements, sleep, and heart event metadata), with diagnostic tables, charts, and a MyFitnessPal comparison. Some device- and scope-dependent endpoints may be unavailable.

**Privacy:** Use an untracked `.env` in the repository root. OAuth codes are entered via a hidden prompt; no credentials are stored in this notebook.

## 1. Imports, date configuration and `.env`

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from dotenv import load_dotenv
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/"fitness_health"/"fitness_analysis.py").exists()), None)
if ROOT is None: raise RuntimeError("Run inside sandbox-scripts repository")
load_dotenv(ROOT/".env", override=False)
sys.path.insert(0, str(ROOT / "fitness_health"))
from fitness_analysis import load_mfp, load_withings, join_data, monthly_summary, forecast, lean_goal_table
START_DATE, END_DATE = "2026-04-01", "2026-10-20"
OUTPUT_DIR = ROOT/"fitness_health"/"withings"/"data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams["figure.figsize"] = (12, 4)
pd.set_option("display.max_columns", 30)
print("Output:", OUTPUT_DIR)# High-contrast chart styling: readable in Jupyter light/dark UI and exports
from cycler import cycler
PLOT_COLORS = ["#1356A2", "#E66B0A", "#147D64", "#A12A50", "#7546B4", "#596775"]
plt.style.use("default")
plt.rcParams.update({
    "figure.facecolor": "#FFFFFF", "axes.facecolor": "#FFFFFF",
    "savefig.facecolor": "#FFFFFF",
    "axes.prop_cycle": cycler(color=PLOT_COLORS),
    "text.color": "#17212B", "axes.labelcolor": "#17212B",
    "axes.edgecolor": "#425466", "xtick.color": "#243447",
    "ytick.color": "#243447", "font.size": 11,
    "axes.titlesize": 14, "axes.titleweight": "bold",
    "axes.labelsize": 11, "legend.fontsize": 10,
    "lines.linewidth": 2.4, "lines.markersize": 6,
    "axes.grid": True, "grid.color": "#DCE3EA",
    "grid.alpha": 0.65, "grid.linewidth": 0.7,
    "figure.figsize": (12, 5.5),
    "figure.dpi": 115,
})


## 2. Withings OAuth — reuse existing `api` if already signed in

In [ ]:
from withings_api import WithingsAuth, WithingsApi
from withings_api.common import AuthScope
if "api" not in globals():
    CLIENT_ID, CLIENT_SECRET = os.getenv("CLIENT_ID"), os.getenv("CLIENT_SECRET")
    if not CLIENT_ID or not CLIENT_SECRET:
        raise ValueError("Missing CLIENT_ID or CLIENT_SECRET in root .env")
    auth = WithingsAuth(client_id=CLIENT_ID, consumer_secret=CLIENT_SECRET,
        callback_uri="http://localhost:5000/get_token",
        scope=(AuthScope.USER_ACTIVITY, AuthScope.USER_METRICS, AuthScope.USER_INFO))
    print("Authorize here:", auth.get_authorize_url())
    import getpass
    from urllib.parse import urlparse, parse_qs
    redirected_url = getpass.getpass("Paste redirect URL (hidden): ")
    code = parse_qs(urlparse(redirected_url.strip()).query).get("code", [])
    if not code: raise ValueError("No OAuth code in redirect URL")
    api = WithingsApi(auth.get_credentials(code[0]))
    del redirected_url, code
    print("Authenticated")
else:
    print("Using already authenticated api session")

## 3. Shared pagination, export, and plotting utilities

In [ ]:
def body_of(response):
    if hasattr(response, "model_dump"): response = response.model_dump()
    elif hasattr(response, "dict"): response = response.dict()
    if not isinstance(response, dict): return {}
    return response.get("body", response)

def query_pages(path, action, params, keys=("series",), limit=100):
    rows, offset, offsets = [], None, set()
    for page in range(limit):
        request = dict(params, action=action)
        if offset is not None: request["offset"] = offset
        body = body_of(api.request(path=path, params=request))
        batch = next((body[k] for k in keys if isinstance(body.get(k), list)), [])
        rows.extend(batch)
        if not body.get("more"): break
        next_offset = body.get("offset")
        if next_offset is None or str(next_offset) in offsets:
            warnings.warn(f"Pagination ended early for {action} (missing/repeated offset)")
            break
        offsets.add(str(next_offset)); offset = next_offset
        time.sleep(.2)
    return rows

def chunks(window=80):
    start, end = pd.Timestamp(START_DATE), pd.Timestamp(END_DATE)
    while start <= end:
        stop = min(start + pd.Timedelta(days=window-1), end)
        yield start.strftime("%Y-%m-%d"), stop.strftime("%Y-%m-%d")
        start = stop + pd.Timedelta(days=1)

def export(name, rows):
    frame = pd.json_normalize(rows)
    if frame.empty:
        print(f"{name}: no records"); return frame
    if "date" in frame: frame = frame.drop_duplicates().sort_values("date")
    frame.to_csv(OUTPUT_DIR/f"withings_{name}.csv", index=False)
    print(f"{name}: {len(frame)} records"); display(frame.head(10))
    return frame

def safe(name, fetch):
    try: return fetch()
    except Exception as exc:
        warnings.warn(f"{name}: unavailable ({type(exc).__name__}: {exc})")
        return pd.DataFrame()

def plot_time(frame, x, ys, title, ylabel=None):
    ys = [y for y in ys if y in frame.columns]
    if frame.empty or not ys or x not in frame: return
    work = frame.copy()
    for y in ys: work[y] = pd.to_numeric(work[y], errors="coerce")
    work.plot(x=x, y=ys, title=title)
    if ylabel: plt.ylabel(ylabel)
    plt.tight_layout(); plt.show()

## 4. Daily activity — total calories, active calories, steps, distance, intensity, heart rate

**Total calories already include physical activity.** Never add workout calories or active calories to this total.

In [ ]:
ACTIVITY_FIELDS = ["steps","distance","elevation","soft","moderate","intense","active",
 "calories","totalcalories","hr_average","hr_min","hr_max",
 "hr_zone_0","hr_zone_1","hr_zone_2","hr_zone_3"]
def get_activity():
    rows=[]
    for start,end in chunks():
        rows += query_pages(api.PATH_V2_MEASURE,"getactivity",
            {"startdateymd":start,"enddateymd":end,
             "data_fields":",".join(ACTIVITY_FIELDS)}, keys=("activities",))
    return export("daily_activity", rows)
activity_df = safe("Activity", get_activity)
plot_time(activity_df,"date",["totalcalories","calories"],"Daily total and active calories","kcal/day")
plot_time(activity_df,"date",["steps"],"Daily steps","steps/day")
plot_time(activity_df,"date",["hr_average","hr_min","hr_max"],"Activity heart-rate metrics","bpm")
if not activity_df.empty:
    display(activity_df.describe(include="all").T)

## 5. Workouts — exercise frequency, duration, energy and intensity

In [ ]:
WORKOUT_FIELDS = ["calories","effduration","intensity","distance","steps","hr_average",
 "hr_min","hr_max","hr_zone_0","hr_zone_1","hr_zone_2","hr_zone_3"]
def get_workouts():
    rows=[]
    for start,end in chunks():
        rows += query_pages(api.PATH_V2_MEASURE,"getworkouts",
            {"startdateymd":start,"enddateymd":end,
             "data_fields":",".join(WORKOUT_FIELDS)})
    return export("workouts",rows)
workouts_df = safe("Workouts",get_workouts)
if not workouts_df.empty:
    if "calories" in workouts_df:
        pd.to_numeric(workouts_df.calories,errors="coerce").hist(bins=25)
        plt.title("Workout estimated calorie distribution");plt.xlabel("kcal");plt.tight_layout();plt.show()
    if "startdate" in workouts_df:
        w = pd.to_datetime(workouts_df.startdate,unit="s",errors="coerce")
        w.dt.to_period("W").value_counts().sort_index().plot(kind="bar",title="Sessions each week")
        plt.tight_layout();plt.show()
    display(workouts_df.describe(include="all").T)

## 6. Body measurements — weight, body composition, blood pressure (device dependent)

The nested `measures` column retains original Withings type and unit codes.

In [ ]:
def get_measures():
    start = int(pd.Timestamp(START_DATE,tz="UTC").timestamp())
    end = int((pd.Timestamp(END_DATE,tz="UTC")+pd.Timedelta(days=1)).timestamp())
    return export("measure_groups",query_pages(api.PATH_MEASURE,"getmeas",
        {"startdate":start,"enddate":end,"category":1},keys=("measuregrps",)))
measure_df = safe("Body measurements",get_measures)
if not measure_df.empty and "date" in measure_df:
    dates = pd.to_datetime(measure_df.date,unit="s",errors="coerce")
    dates.dt.to_period("M").value_counts().sort_index().plot(kind="bar",title="Measurements per month")
    plt.tight_layout();plt.show()

## 7. Sleep summaries — sleep score, stages, duration, heart rate

In [ ]:
SLEEP_FIELDS = ["sleep_score","wakeupcount","deepsleepduration",
 "lightsleepduration","remsleepduration","wakeupduration",
 "hr_average","hr_min","hr_max"]
def get_sleep():
    rows=[]
    for start,end in chunks():
        rows += query_pages(api.PATH_V2_SLEEP,"getsummary",
            {"startdateymd":start,"enddateymd":end,
             "data_fields":",".join(SLEEP_FIELDS)})
    return export("sleep_summary",rows)
sleep_df=safe("Sleep summary",get_sleep)
if not sleep_df.empty:
    cols=[x for x in sleep_df if "sleep_score" in x or "sleepduration" in x]
    for col in cols:
        series=pd.to_numeric(sleep_df[col],errors="coerce")
        if series.notna().any():
            series.plot(title=f"Sleep: {col}");plt.tight_layout();plt.show()

## 8. Sleep segments — optional detailed epochs

In [ ]:
def get_sleep_epochs():
    rows=[]
    for start,end in chunks(window=14):
        begin=int(pd.Timestamp(start,tz="UTC").timestamp())
        finish=int((pd.Timestamp(end,tz="UTC")+pd.Timedelta(days=1)).timestamp())
        rows += query_pages(api.PATH_V2_SLEEP,"get",
            {"startdate":begin,"enddate":finish})
    return export("sleep_epochs",rows)
epochs_df=safe("Sleep epochs",get_sleep_epochs)
if not epochs_df.empty and "state" in epochs_df:
    epochs_df.state.value_counts().plot(kind="bar",title="Sleep-stage segments")
    plt.tight_layout();plt.show()

## 9. Heart events / ECG metadata — only where supported

This is not a medical interpretation; signal waveform retrieval is intentionally excluded.

In [ ]:
def get_heart():
    begin=int(pd.Timestamp(START_DATE,tz="UTC").timestamp())
    end=int((pd.Timestamp(END_DATE,tz="UTC")+pd.Timedelta(days=1)).timestamp())
    return export("heart_events",query_pages(api.PATH_V2_HEART,"list",
        {"startdate":begin,"enddate":end}))
heart_df=safe("Heart events",get_heart)
if not heart_df.empty and "date" in heart_df:
    hd=pd.to_datetime(heart_df.date,unit="s",errors="coerce")
    hd.dt.to_period("M").value_counts().sort_index().plot(kind="bar",title="Heart events per month")
    plt.tight_layout();plt.show()

## 10. Integrate MyFitnessPal — recorded food, weight and Withings energy

In [ ]:
mfp_daily,mfp_weight=load_mfp(ROOT/"fitness_health"/"myfitnesspal"/"data")
wearable=load_withings(OUTPUT_DIR)
merged=join_data(mfp_daily,mfp_weight,wearable)
display(monthly_summary(merged))
display(merged.tail(10))
merged.to_csv(OUTPUT_DIR/"withings_mfp_combined.csv",index=False)
plot_time(merged,"date",["weight_kg","weight_7d_mean"],"Recorded weight history","kg")
plot_time(merged,"date",["tdee_calories"],"Estimated total expenditure","kcal/day")
if "observed_deficit_kcal" in merged.columns:
    plot_time(merged,"date",["observed_deficit_kcal"],
              "Recorded-day deficit (exclude absent/incomplete food logs)","kcal")

## 11. Weight projections, TDEE sensitivity, maintenance and fat-free-mass goals

Model: 1,800 kcal on baseline days plus two 600-kcal meal extras per week; TDEE reduces 22 kcal/day for each kg lost. Scenario dates are not guarantees.

In [ ]:
fig,ax=plt.subplots()
for tdee in (2450,2600,2750):
    trajectory,milestones=forecast(start_weight=86.8,start_date=END,tdee=tdee)
    ax.plot(trajectory.date,trajectory.weight_kg,label=f"Initial TDEE {tdee}",color=PLOT_COLORS[[2450,2600,2750].index(tdee)],linewidth=2.6)
    print("TDEE",tdee);display(milestones)
for goal in (84,82,80,78,76):
    ax.axhline(goal,linestyle=":",linewidth=.5,color="#A2AAB4")
ax.legend();ax.set_title("Weight-loss sensitivity");ax.set_ylabel("kg")
plt.tight_layout();plt.show()
maintenance = pd.DataFrame({"weight_kg":[84,82,80,78,76]})
maintenance["kcal_per_day"]=2600+22*(maintenance.weight_kg-86.8)
display(maintenance.round())
goals=lean_goal_table(dexa_ffm=78.021,height_m=1.77)
display(goals.round(2))
fig,ax=plt.subplots()
for _,row in goals.iterrows():
    rates=np.array([.5,1,1.5,2,3])
    ax.plot(rates,row.additional_ffm_kg/rates,marker="o",label=row.goal)
ax.set_title("Time to gain required FFM by annual rate (scenario only)")
ax.set_xlabel("Assumed annual gain, kg FFM");ax.set_ylabel("Years");ax.legend()
plt.tight_layout();plt.show()

## 12. Outputs and known limitations

All CSVs are saved to `withings/data/`. Withings energy expenditure is estimated; food omissions are unknown intake; DEXA and bioimpedance are not interchangeable. OAuth/API calls were not executed as part of this repository change and need validation in your authenticated environment.